# Interactive UMAPs for CellTypist on the Human Immune Health Atlas

In [1]:
from pathlib import Path
import html

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from scipy.ndimage import gaussian_filter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Update DATA_ROOT if you are running this notebook on another computer.
DATA_ROOT = Path(
    "/Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE"
)
DATA_DIR = DATA_ROOT / "iha_celltypist"
# Use the original prepared dataset and the baseline outputs already in iha_celltypist.
# These three files belong together; the masking notebook is not required for baseline UMAPs.
PREPARED_H5AD = DATA_DIR / "iha_expression_prepared.h5ad"
PREDICTIONS_CSV = DATA_DIR / "baseline_predictions.csv"
DECISION_MATRIX_CSV = DATA_DIR / "baseline_decision_matrix.csv"

OUT_DIR = DATA_DIR / "umap_out"
INTERACTIVE_DIR = OUT_DIR / "interactive"

CELL_KEY = "soma_joinid"
AGE_COL = "age_group"
ATLAS_COL = "cell_type"
PRED_COL = "predicted_labels"

N_HVG = 2_000
N_PCS = 50
N_NEIGHBORS = 15
MIN_DIST = 0.3
SEED = 42
POINT_SIZE = 3
OUTLINE_MIN_CELLS = 100
OUTLINE_MASS = 0.80
OUTLINE_BINS = 120
OUTLINE_SIGMA = 1.5
FOCUS_SUMMARY_LABELS = 8
SHOW_INLINE = False  # Set True to render HTML figures in the notebook output.

OUT_DIR.mkdir(parents=True, exist_ok=True)
INTERACTIVE_DIR.mkdir(parents=True, exist_ok=True)

for required_path in (PREPARED_H5AD, PREDICTIONS_CSV, DECISION_MATRIX_CSV):
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required input not found: {required_path}. "
            "Run the gene-masking notebook and check the paths above."
        )

print(f"Prepared input: {PREPARED_H5AD}")
print(f"Unmasked predictions: {PREDICTIONS_CSV}")
print(f"Decision matrix: {DECISION_MATRIX_CSV}")
print(f"HTML output folder: {INTERACTIVE_DIR}")


Prepared input: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/iha_expression_prepared.h5ad
Unmasked predictions: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/baseline_predictions.csv
Decision matrix: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/baseline_decision_matrix.csv
HTML output folder: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/umap_out/interactive


## 1. Load data and align predictions by cell ID

The AnnData object supplies expression and atlas metadata. The prediction and decision-score tables are aligned by `soma_joinid`, not by row order. The outputs from the updated masking notebook are used so the predictions correspond to this model-gene prepared input and its normalization.

In [2]:
def read_aligned_csv(csv_path, cell_ids):
    table = pd.read_csv(csv_path, dtype={CELL_KEY: str})
    key = CELL_KEY if CELL_KEY in table.columns else table.columns[0]
    if table[key].isna().any() or table[key].duplicated().any():
        raise ValueError(f"{csv_path.name} has missing or duplicate cell IDs.")
    table = table.set_index(key)
    table.index = table.index.astype(str)

    missing = cell_ids.difference(table.index)
    extra = table.index.difference(cell_ids)
    if len(missing) or len(extra):
        raise ValueError(
            f"Cell IDs in {csv_path.name} do not match the H5AD. "
            f"Missing: {len(missing):,}; extra: {len(extra):,}."
        )
    return table.loc[cell_ids]


adata = ad.read_h5ad(PREPARED_H5AD)
if CELL_KEY not in adata.obs.columns:
    raise KeyError(f"Prepared H5AD must contain obs[{CELL_KEY!r}].")
for column in (AGE_COL, ATLAS_COL):
    if column not in adata.obs.columns:
        raise KeyError(f"Prepared H5AD is missing obs[{column!r}].")

cell_ids_numeric = pd.to_numeric(adata.obs[CELL_KEY], errors="raise")
if cell_ids_numeric.isna().any():
    raise ValueError("Prepared H5AD contains missing soma_joinid values.")
cell_ids = pd.Index(cell_ids_numeric.astype("int64").astype(str), name=CELL_KEY)
if not cell_ids.is_unique:
    raise ValueError("Prepared H5AD contains duplicate soma_joinid values.")
adata.obs_names = cell_ids
adata.obs[CELL_KEY] = cell_ids.to_numpy()

for column in (AGE_COL, ATLAS_COL):
    if adata.obs[column].isna().any():
        raise ValueError(f"Prepared H5AD contains missing values in {column!r}.")
    adata.obs[column] = adata.obs[column].astype(str)

prediction_table = read_aligned_csv(PREDICTIONS_CSV, cell_ids)
if PRED_COL not in prediction_table.columns:
    raise KeyError(
        f"{PRED_COL!r} is missing from {PREDICTIONS_CSV.name}; "
        f"available columns: {prediction_table.columns.tolist()}"
    )
if prediction_table[PRED_COL].isna().any():
    raise ValueError("The prediction table contains missing predicted labels.")
adata.obs[PRED_COL] = prediction_table[PRED_COL].astype(str).to_numpy()

score_table = read_aligned_csv(DECISION_MATRIX_CSV, cell_ids)
scores = score_table.apply(pd.to_numeric, errors="coerce").dropna(axis=1, how="all")
if scores.shape[1] < 2 or not np.isfinite(scores.to_numpy(dtype=float)).all():
    raise ValueError("Decision matrix must contain at least two complete numeric class-score columns.")
score_values = scores.to_numpy(dtype=float)
top_two = np.partition(score_values, kth=score_values.shape[1] - 2, axis=1)[:, -2:]
adata.obs["top_decision_score"] = top_two.max(axis=1)
adata.obs["decision_margin"] = top_two.max(axis=1) - top_two.min(axis=1)

print(f"Loaded {adata.n_obs:,} cells and {adata.n_vars:,} model genes.")
print(f"Atlas labels: {adata.obs[ATLAS_COL].nunique():,}; CellTypist classes observed: {adata.obs[PRED_COL].nunique():,}.")
print("Prediction and score rows matched to every cell by soma_joinid.")


Loaded 54,225 cells and 61,497 model genes.
Atlas labels: 36; CellTypist classes observed: 61.
Prediction and score rows matched to every cell by soma_joinid.


## 2. Add descriptive label and score fields, then compute one shared UMAP

**Not the usual prediction** means the cell’s CellTypist prediction differs from the most frequent prediction observed for cells with the same original atlas `cell_type`, calculated across all age groups. It is a reference to the dominant pattern in this dataset, not a correctness label or an independent biological mapping.

The decision matrix provides model scores. We show the highest class score and the difference between the highest and second-highest scores. The margin is a useful measure of how separated the top two scores are, but it should not be described as calibrated probability or certainty.

UMAP is computed once on all selected cells using highly variable genes from the prepared model-gene input. Age groups are facets of this shared embedding.

In [3]:
# Define the usual prediction across all age groups for each original atlas label.
dominant_prediction = (
    adata.obs.groupby(ATLAS_COL, observed=True)[PRED_COL]
    .agg(lambda labels: labels.value_counts().index[0])
)
adata.obs["usual_prediction_for_atlas_type"] = adata.obs[ATLAS_COL].map(dominant_prediction)
adata.obs["usual_prediction_status"] = np.where(
    adata.obs[PRED_COL] == adata.obs["usual_prediction_for_atlas_type"],
    "usual prediction",
    "not the usual prediction",
)

usual_table = dominant_prediction.rename("usual_prediction").reset_index()
usual_table.to_csv(OUT_DIR / "usual_prediction_by_atlas_type.csv", index=False)

# Give age groups a human-readable order when values use the expected labels.
def age_group_key(value):
    normalized = str(value).replace("–", "-").replace("—", "-").strip()
    if normalized.startswith("<"):
        return (0, normalized)
    if "18" in normalized and "45" in normalized:
        return (1, normalized)
    if normalized.startswith(">"):
        return (2, normalized)
    return (3, normalized)

AGE_GROUPS = sorted(adata.obs[AGE_COL].unique(), key=age_group_key)

# Compute one joint embedding. No donor batch key or donor-specific filtering is used.
work = adata.copy()
sc.pp.filter_genes(work, min_cells=10)
n_hvg = min(N_HVG, work.n_vars)
if n_hvg < 2:
    raise ValueError("Not enough genes remain to compute a UMAP embedding.")
sc.pp.highly_variable_genes(
    work,
    n_top_genes=n_hvg,
    flavor="seurat",
)
work = work[:, work.var["highly_variable"]].copy()
work.X = work.X.astype(np.float32)
n_pcs = min(N_PCS, work.n_vars - 1, work.n_obs - 1)
sc.pp.pca(work, n_comps=n_pcs, random_state=SEED)
sc.pp.neighbors(work, n_neighbors=N_NEIGHBORS, n_pcs=n_pcs, random_state=SEED)
sc.tl.umap(work, min_dist=MIN_DIST, random_state=SEED)

coords = pd.DataFrame(
    work.obsm["X_umap"],
    index=adata.obs_names,
    columns=["umap1", "umap2"],
)
plot_data = coords.join(adata.obs)
if not plot_data.index.equals(adata.obs_names):
    raise AssertionError("UMAP coordinates lost their cell-ID alignment.")

plot_data.to_csv(OUT_DIR / "joint_umap_coordinates_and_metadata.csv", index_label=CELL_KEY)
print(f"Computed one shared embedding for {len(plot_data):,} cells.")
print(f"Age groups: {AGE_GROUPS}")


/Users/artsvetkova/miniconda3/envs/sc_immune/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Computed one shared embedding for 54,225 cells.
Age groups: ['<18', '18-45', '>45']


## 3. Interactive UMAP explorer: atlas labels, model labels, unusual predictions, and scores

The dropdown changes the color layer without changing coordinates. Use the legend to hide or isolate categories. Hover over a point to see its original atlas label, model prediction, `soma_joinid`, age group, and score summaries.

In [4]:
def make_palette(values):
    categories = sorted(pd.Series(values).astype(str).unique())
    color_list = []
    for cmap_name, n_colors in (("tab20", 20), ("tab20b", 20), ("tab20c", 20), ("Set3", 12)):
        cmap = plt.get_cmap(cmap_name)
        color_list.extend([to_hex(cmap(i)) for i in range(n_colors)])
    if len(categories) > len(color_list):
        extra = plt.get_cmap("hsv")
        color_list.extend(
            [to_hex(extra(x)) for x in np.linspace(0, 1, len(categories) - len(color_list), endpoint=False)]
        )
    return dict(zip(categories, color_list))


ATLAS_COLORS = make_palette(plot_data[ATLAS_COL])
PRED_COLORS = make_palette(plot_data[PRED_COL])
USUAL_COLORS = {
    "usual prediction": "#bdbdbd",
    "not the usual prediction": "#e6550d",
}


def hover_text(frame):
    rows = []
    for cell_id, row in frame.iterrows():
        rows.append(
            "<b>Atlas:</b> " + html.escape(str(row[ATLAS_COL]))
            + "<br><b>CellTypist:</b> " + html.escape(str(row[PRED_COL]))
            + "<br><b>Usual prediction for this atlas type:</b> "
            + html.escape(str(row["usual_prediction_for_atlas_type"]))
            + "<br><b>Status:</b> " + html.escape(str(row["usual_prediction_status"]))
            + f"<br><b>Top class score:</b> {row['top_decision_score']:.4f}"
            + f"<br><b>Top1–top2 score margin:</b> {row['decision_margin']:.4f}"
            + "<br><b>Age group:</b> " + html.escape(str(row[AGE_COL]))
            + "<br><b>soma_joinid:</b> " + html.escape(str(cell_id))
        )
    return np.asarray(rows, dtype=object)


def interactive_umap_explorer(frame, output_path):
    facets = AGE_GROUPS
    masks = [(frame[AGE_COL].to_numpy() == group) for group in facets]
    fig = make_subplots(
        rows=1,
        cols=len(facets),
        shared_xaxes="all",
        shared_yaxes="all",
        horizontal_spacing=0.035,
        subplot_titles=[f"{AGE_COL}: {g} ({m.sum():,} cells)" for g, m in zip(facets, masks)],
    )
    x = frame["umap1"].to_numpy(dtype=np.float32)
    y = frame["umap2"].to_numpy(dtype=np.float32)
    layers = [
        ("Atlas cell type", "cat", ATLAS_COL, ATLAS_COLORS),
        ("CellTypist prediction", "cat", PRED_COL, PRED_COLORS),
        ("Not the usual prediction", "cat", "usual_prediction_status", USUAL_COLORS),
        # ("Top class decision score", "num", "top_decision_score", "Viridis"),
        ("Top1–top2 score margin", "num", "decision_margin", "Viridis"),
    ]
    trace_layer = []

    for layer_index, (layer_name, kind, column, colors) in enumerate(layers):
        values = frame[column]
        visible = layer_index == 0
        if kind == "cat":
            category_counts = values.value_counts()
            for category in sorted(category_counts.index):
                in_category = (values == category).to_numpy()
                for facet_index, facet_mask in enumerate(masks):
                    selected = in_category & facet_mask
                    fig.add_trace(
                        go.Scattergl(
                            x=x[selected],
                            y=y[selected],
                            mode="markers",
                            marker=dict(size=POINT_SIZE, color=colors.get(category, "#999999")),
                            name=f"{category} ({category_counts[category]:,})",
                            legendgroup=f"{layer_index}:{category}",
                            showlegend=facet_index == 0,
                            hoverinfo="skip",
                            visible=visible,
                        ),
                        row=1,
                        col=facet_index + 1,
                    )
                    trace_layer.append(layer_index)
        else:
            numeric = pd.to_numeric(values, errors="raise").to_numpy(dtype=float)
            low, high = np.nanpercentile(numeric, [1, 99])
            if not np.isfinite(low) or not np.isfinite(high):
                raise ValueError(f"No finite values available for {column!r}.")
            if np.isclose(low, high):
                high = low + 1e-9
            for facet_index, facet_mask in enumerate(masks):
                selected_indices = np.flatnonzero(facet_mask & np.isfinite(numeric))
                selected_indices = selected_indices[np.argsort(-numeric[selected_indices])]
                fig.add_trace(
                    go.Scattergl(
                        x=x[selected_indices],
                        y=y[selected_indices],
                        mode="markers",
                        marker=dict(
                            size=POINT_SIZE,
                            color=numeric[selected_indices],
                            colorscale=colors,
                            cmin=low,
                            cmax=high,
                            showscale=facet_index == len(masks) - 1,
                            colorbar=dict(title=layer_name, len=0.78),
                        ),
                        showlegend=False,
                        hoverinfo="skip",
                        visible=visible,
                    ),
                    row=1,
                    col=facet_index + 1,
                )
                trace_layer.append(layer_index)

    hover = hover_text(frame)
    for facet_index, facet_mask in enumerate(masks):
        fig.add_trace(
            go.Scattergl(
                x=x[facet_mask],
                y=y[facet_mask],
                mode="markers",
                text=hover[facet_mask],
                marker=dict(size=POINT_SIZE + 3, color="rgba(0,0,0,0.01)"),
                hovertemplate="%{text}<extra></extra>",
                showlegend=False,
                name="cell details",
            ),
            row=1,
            col=facet_index + 1,
        )
        trace_layer.append("hover")

    buttons = []
    for layer_index, (layer_name, *_rest) in enumerate(layers):
        buttons.append(
            dict(
                label=layer_name,
                method="update",
                args=[
                    {"visible": [tag == layer_index or tag == "hover" for tag in trace_layer]},
                    {"legend.title.text": layer_name},
                ],
            )
        )

    fig.update_layout(
        title=dict(text="One shared UMAP across age groups", x=0.5),
        template="simple_white",
        height=730,
        width=520 * len(facets) + 380,
        hovermode="closest",
        legend=dict(itemsizing="constant", font=dict(size=9), title=dict(text=layers[0][0])),
        updatemenus=[dict(buttons=buttons, direction="down", showactive=True, x=0, xanchor="left", y=1.16, yanchor="top")],
        annotations=list(fig.layout.annotations) + [dict(text="Color by:", x=0, xref="paper", xanchor="right", y=1.14, yref="paper", showarrow=False)],
        margin=dict(t=130, l=65, r=180, b=40),
    )
    fig.update_xaxes(showticklabels=False, showgrid=False, ticks="")
    for facet_index in range(len(facets)):
        x_axis = "x" if facet_index == 0 else f"x{facet_index + 1}"
        fig.update_yaxes(
            showticklabels=False,
            showgrid=False,
            ticks="",
            scaleanchor=x_axis,
            scaleratio=1,
            row=1,
            col=facet_index + 1,
        )
    fig.write_html(
        output_path,
        include_plotlyjs=True,
        config={"scrollZoom": True, "displaylogo": False, "toImageButtonOptions": {"format": "png", "scale": 3}},
    )
    print(f"Saved interactive explorer: {output_path}")
    if SHOW_INLINE:
        fig.show()
    return fig


main_umap_path = INTERACTIVE_DIR / "joint_umap_color_explorer_by_age.html"
main_umap = interactive_umap_explorer(plot_data, main_umap_path)


Saved interactive explorer: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/umap_out/interactive/joint_umap_color_explorer_by_age.html


## 4. Interactive atlas-point / CellTypist-contour view

This companion view colors points by the original atlas label and draws a smooth density contour for each observed CellTypist class. A contour encloses the central `OUTLINE_MASS` fraction of that class’s cells in the joint UMAP. It is a descriptive density boundary, not a hard cluster boundary. Hover over a contour to identify the model class; use the legend to show or hide classes.

In [5]:
def density_boundary(points, x_range, y_range, bins=OUTLINE_BINS, sigma=OUTLINE_SIGMA, mass=OUTLINE_MASS):
    histogram, x_edges, y_edges = np.histogram2d(
        points[:, 0],
        points[:, 1],
        bins=bins,
        range=[x_range, y_range],
    )
    density = gaussian_filter(histogram.astype(float), sigma=sigma)
    sorted_density = np.sort(density.ravel())[::-1]
    cumulative = np.cumsum(sorted_density)
    if cumulative[-1] <= 0:
        return None
    level_index = min(np.searchsorted(cumulative, mass * cumulative[-1]), len(sorted_density) - 1)
    level = float(sorted_density[level_index])
    if level <= 0:
        return None
    x_centers = (x_edges[:-1] + x_edges[1:]) / 2
    y_centers = (y_edges[:-1] + y_edges[1:]) / 2
    return x_centers, y_centers, density.T, level


x_min, x_max = plot_data["umap1"].min(), plot_data["umap1"].max()
y_min, y_max = plot_data["umap2"].min(), plot_data["umap2"].max()
x_padding = (x_max - x_min) * 0.03
y_padding = (y_max - y_min) * 0.03
x_range = (x_min - x_padding, x_max + x_padding)
y_range = (y_min - y_padding, y_max + y_padding)

contour_fig, contour_ax = plt.subplots()

outline_fig = make_subplots(
    rows=1,
    cols=len(AGE_GROUPS),
    shared_xaxes="all",
    shared_yaxes="all",
    horizontal_spacing=0.035,
    subplot_titles=[f"{AGE_COL}: {group}" for group in AGE_GROUPS],
)

for facet_index, group in enumerate(AGE_GROUPS):
    facet = plot_data.loc[plot_data[AGE_COL] == group]
    for atlas_type in sorted(facet[ATLAS_COL].unique()):
        rows = facet.loc[facet[ATLAS_COL] == atlas_type]
        outline_fig.add_trace(
            go.Scattergl(
                x=rows["umap1"],
                y=rows["umap2"],
                mode="markers",
                marker=dict(size=POINT_SIZE, color=ATLAS_COLORS[atlas_type], opacity=0.8),
                name=f"Atlas: {atlas_type}",
                legendgroup=f"atlas:{atlas_type}",
                showlegend=facet_index == 0,
                hoverinfo="skip",
            ),
            row=1,
            col=facet_index + 1,
        )

    for model_label in sorted(facet[PRED_COL].unique()):
        rows = facet.loc[facet[PRED_COL] == model_label]
        if len(rows) < OUTLINE_MIN_CELLS:
            continue
        boundary = density_boundary(
            rows[["umap1", "umap2"]].to_numpy(),
            x_range=x_range,
            y_range=y_range,
        )
        if boundary is None:
            continue
        xc, yc, density, level = boundary
        # Convert the density boundary to line coordinates. A Plotly Contour trace
        # accepts hover across its whole grid, not only along the visible contour;
        # explicit line traces keep each hover label attached to its real boundary.
        contour_set = contour_ax.contour(xc, yc, density, levels=[level])
        line_segments = contour_set.allsegs[0]
        contour_ax.clear()

        legend_written = False
        for segment in line_segments:
            if len(segment) < 2:
                continue
            outline_fig.add_trace(
                go.Scatter(
                    x=segment[:, 0],
                    y=segment[:, 1],
                    mode="lines",
                    line=dict(color=PRED_COLORS[model_label], width=2),
                    name=f"CellTypist: {model_label} (n={len(rows):,})",
                    legendgroup=f"model:{model_label}",
                    showlegend=(facet_index == 0 and not legend_written),
                    hovertemplate=(
                        f"CellTypist class: {html.escape(model_label)}<br>"
                        f"Cells in age group: {len(rows):,}<extra></extra>"
                    ),
                ),
                row=1,
                col=facet_index + 1,
            )
            legend_written = True

plt.close(contour_fig)

outline_fig.update_layout(
    title=dict(text="Atlas labels as points; CellTypist classes as density contours", x=0.5),
    template="simple_white",
    height=800,
    width=560 * len(AGE_GROUPS) + 420,
    legend=dict(font=dict(size=9), itemsizing="constant", tracegroupgap=3),
    margin=dict(t=100, l=60, r=300, b=40),
)
outline_fig.update_xaxes(range=x_range, showticklabels=False, showgrid=False, ticks="")
for facet_index in range(len(AGE_GROUPS)):
    x_axis = "x" if facet_index == 0 else f"x{facet_index + 1}"
    outline_fig.update_yaxes(
        range=y_range,
        showticklabels=False,
        showgrid=False,
        ticks="",
        scaleanchor=x_axis,
        scaleratio=1,
        row=1,
        col=facet_index + 1,
    )
outline_path = INTERACTIVE_DIR / "atlas_points_celltypist_class_contours.html"
outline_fig.write_html(
    outline_path,
    include_plotlyjs=True,
    config={"scrollZoom": True, "displaylogo": False, "toImageButtonOptions": {"format": "png", "scale": 3}},
)
print(f"Saved interactive contour view: {outline_path}")
if SHOW_INLINE:
    outline_fig.show()


Saved interactive contour view: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/umap_out/interactive/atlas_points_celltypist_class_contours.html


## 5. Interactive focus views

Each focus HTML has an age-group facet of the same joint coordinates. Select one atlas type to see how CellTypist labels its cells, or select one CellTypist class to see which atlas types it contains. Other cells remain grey. For readability, only the most frequent eight labels appear in the title summary; less frequent labels are shown in grey, while hover still reports the cell’s exact label.

In [6]:
def focus_explorer(frame, focus_col, color_col, color_palette, title, output_name):
    focus_counts = frame[focus_col].value_counts()
    focus_values = focus_counts.index.tolist()
    facets = AGE_GROUPS
    masks = [(frame[AGE_COL].to_numpy() == group) for group in facets]
    x = frame["umap1"].to_numpy(dtype=np.float32)
    y = frame["umap2"].to_numpy(dtype=np.float32)
    hover = hover_text(frame)

    fig = make_subplots(
        rows=1,
        cols=len(facets),
        shared_xaxes="all",
        shared_yaxes="all",
        horizontal_spacing=0.035,
        subplot_titles=[f"{AGE_COL}: {group}" for group in facets],
    )
    trace_role = []

    # Grey context points are shared by every dropdown option.
    for facet_index, facet_mask in enumerate(masks):
        fig.add_trace(
            go.Scattergl(
                x=x[facet_mask],
                y=y[facet_mask],
                mode="markers",
                marker=dict(size=POINT_SIZE, color="#e5e5e5"),
                hoverinfo="skip",
                showlegend=False,
                name="Other cells",
            ),
            row=1,
            col=facet_index + 1,
        )
        trace_role.append("background")

    title_by_focus = {}
    for focus_index, focus_value in enumerate(focus_values):
        selected = (frame[focus_col].to_numpy() == focus_value)
        selected_frame = frame.loc[selected]
        shares = selected_frame[color_col].value_counts(normalize=True)
        common_labels = shares.head(FOCUS_SUMMARY_LABELS).index.tolist()
        rare_n = int(selected_frame[color_col].isin(shares.index.difference(common_labels)).sum())
        swatches = [
            f"<span style='color:{color_palette[label]}'>●</span> {html.escape(str(label))} {shares[label]:.1%}"
            for label in common_labels
        ]
        if rare_n:
            swatches.append(f"<span style='color:#bdbdbd'>●</span> Other {rare_n / len(selected_frame):.1%}")
        focus_title = (
            f"{html.escape(str(title))}<br><sup>{html.escape(str(focus_value))} "
            f"(n={len(selected_frame):,}) · " + " &nbsp; ".join(swatches) + "</sup>"
        )
        title_by_focus[focus_index] = focus_title

        color_labels = selected_frame[color_col].astype(str).to_numpy()
        color_map = np.array([
            color_palette.get(label, "#888888") if label in common_labels else "#bdbdbd"
            for label in color_labels
        ], dtype=object)
        selected_indices = np.flatnonzero(selected)
        selected_hover = hover[selected]

        for facet_index, facet_mask in enumerate(masks):
            local = facet_mask[selected_indices]
            idx = selected_indices[local]
            if len(idx) == 0:
                continue
            # Keep the top labels colored and group rare labels in grey.
            colors_here = color_map[local]
            fig.add_trace(
                go.Scattergl(
                    x=x[idx],
                    y=y[idx],
                    mode="markers",
                    text=hover[idx],
                    marker=dict(size=POINT_SIZE + 1, color=colors_here),
                    hovertemplate="%{text}<extra></extra>",
                    showlegend=False,
                    name=str(focus_value),
                    visible=focus_index == 0,
                ),
                row=1,
                col=facet_index + 1,
            )
            trace_role.append(focus_index)

    buttons = []
    for focus_index, focus_value in enumerate(focus_values):
        buttons.append(
            dict(
                label=f"{focus_value} ({focus_counts[focus_value]:,})",
                method="update",
                args=[
                    {"visible": [role == "background" or role == focus_index for role in trace_role]},
                    {"title.text": title_by_focus[focus_index]},
                ],
            )
        )

    fig.update_layout(
        title=dict(text=title_by_focus[0], x=0.5),
        template="simple_white",
        height=720,
        width=520 * len(facets) + 320,
        hovermode="closest",
        updatemenus=[dict(buttons=buttons, direction="down", showactive=True, x=0, xanchor="left", y=1.18, yanchor="top")],
        annotations=list(fig.layout.annotations) + [dict(text="Focus on:", x=0, xref="paper", xanchor="right", y=1.16, yref="paper", showarrow=False)],
        margin=dict(t=175, l=65, r=50, b=35),
    )
    fig.update_xaxes(showticklabels=False, showgrid=False, ticks="")
    for facet_index in range(len(facets)):
        x_axis = "x" if facet_index == 0 else f"x{facet_index + 1}"
        fig.update_yaxes(
            showticklabels=False,
            showgrid=False,
            ticks="",
            scaleanchor=x_axis,
            scaleratio=1,
            row=1,
            col=facet_index + 1,
        )

    output_path = INTERACTIVE_DIR / output_name
    fig.write_html(
        output_path,
        include_plotlyjs=True,
        config={"scrollZoom": True, "displaylogo": False, "toImageButtonOptions": {"format": "png", "scale": 3}},
    )
    print(f"Saved focus explorer: {output_path}")
    if SHOW_INLINE:
        fig.show()
    return fig


focus_by_atlas = focus_explorer(
    plot_data,
    ATLAS_COL,
    PRED_COL,
    PRED_COLORS,
    "Focus on an atlas type: CellTypist predictions for its cells",
    "focus_by_atlas_type.html",
)
focus_by_prediction = focus_explorer(
    plot_data,
    PRED_COL,
    ATLAS_COL,
    ATLAS_COLORS,
    "Focus on a CellTypist class: atlas labels among its cells",
    "focus_by_prediction.html",
)

print("All interactive HTML views are ready in:", INTERACTIVE_DIR.resolve())


Saved focus explorer: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/umap_out/interactive/focus_by_atlas_type.html
Saved focus explorer: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/umap_out/interactive/focus_by_prediction.html
All interactive HTML views are ready in: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/umap_out/interactive
